# Phase 2 - Data Generation: Parking Dataset Augmentation

This notebook uses the `P2-00.5_parking_augment.py` module to create and visualize an augmented parking dataset.

**Purpose:**
- Generate augmented parking dataset from PKLot dataset
- Create JSON format annotations
- Apply rotations and other augmentations
- Visualize augmented samples

**Output:** Augmented dataset saved to `data/processed/parking_dataset_augmented/`

In [ ]:
from pathlib import Path
import sys
import random
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import numpy as np

# Add Phase 2 directory to path to import the augmentation module
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "Phase 2" else Path(__file__).parent if '__file__' in globals() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

# Add report directory to path for utilities
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "Phase 2" else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(BASE_DIR / "report"))

# Import report utilities
try:
    from utils_report import save_plot, save_eda_plot, save_dataset_statistics, get_eda_dir
    REPORT_UTILS_AVAILABLE = True
except ImportError:
    print("Warning: Report utilities not available. Plots will not be saved.")
    REPORT_UTILS_AVAILABLE = False

# Import from our parking_augment module (same directory)
# Note: File renamed from P2-00.5_parking_augment.py to P2_00_5_parking_augment.py for Python compatibility
try:
    from P2_00_5_parking_augment import (
        create_augmented_dataset,
        visualize_sample,
        CROP_REGIONS
    )
    print("Successfully imported augmentation module")
except ImportError as e:


## 1. Set Paths

In [ ]:
# Set base directory (project root)
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "Phase 2" else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Input: PKLot dataset (adjust path as needed)
DATA_ROOT = BASE_DIR / "data" / "raw" / "PKLot" / "PKLot"  # Adjust this path to your PKLot dataset location

# Output: Augmented dataset directory
OUTPUT_DIR = BASE_DIR / "data" / "processed" / "parking_dataset_augmented"

# Report directory for saving visualizations
if REPORT_UTILS_AVAILABLE:
    REPORT_DIR = get_eda_dir()
    print(f"Report directory: {REPORT_DIR}")

print(f"BASE_DIR: {BASE_DIR}")
print(f"Input (PKLot): {DATA_ROOT}")
print(f"Input exists: {DATA_ROOT.exists()}")


## 2. Create Augmented Dataset

In [ ]:
# Create the dataset with custom parameters
create_augmented_dataset(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    target_size=(640, 640),      # Output image size
    num_augmentations=3,          # 1 original + 2 augmented versions
    random_seed=42
)

## 3. Visualize Random Samples

In [ ]:
# Show 9 random samples from the dataset
visualize_sample(OUTPUT_DIR, num_samples=9)

# Save the visualization if report utilities are available
if REPORT_UTILS_AVAILABLE:
    fig = plt.gcf()
    if fig is not None:
        save_eda_plot(fig, "phase2_augmented_samples.png")

## 4. Dataset Statistics

In [ ]:
# Load dataset info
with open(OUTPUT_DIR / "dataset_info.json", 'r') as f:
    dataset_info = json.load(f)

print(f"Total images: {dataset_info['num_images']}")
print(f"Image size: {dataset_info['target_size']}")
print(f"Augmentations per original: {dataset_info['num_augmentations']}")

# Count statistics
total_spaces = sum(img['num_spaces'] for img in dataset_info['images'])
total_occupied = sum(img['num_occupied'] for img in dataset_info['images'])

print(f"\nTotal parking spaces: {total_spaces:,}")
print(f"Occupied: {total_occupied:,} ({100*total_occupied/total_spaces:.1f}%)")
print(f"Empty: {total_spaces - total_occupied:,} ({100*(total_spaces-total_occupied)/total_spaces:.1f}%)")

# Save statistics to report directory
if REPORT_UTILS_AVAILABLE:
    stats = {
        "Total images": dataset_info['num_images'],
        "Image size": str(dataset_info['target_size']),
        "Augmentations per original": dataset_info['num_augmentations'],
        "Total parking spaces": f"{total_spaces:,}",
        "Occupied spaces": f"{total_occupied:,}",
        "Occupied percentage": f"{100*total_occupied/total_spaces:.1f}%",
        "Empty spaces": f"{total_spaces - total_occupied:,}",
        "Empty percentage": f"{100*(total_spaces-total_occupied)/total_spaces:.1f}%"
    }
    save_dataset_statistics(stats, "phase2_augmented_dataset_statistics.txt")

## 5. Examine Single Sample

In [ ]:
# Load a single sample to see the data structure
images_dir = OUTPUT_DIR / "images"
annotations_dir = OUTPUT_DIR / "annotations"

# Get first annotation
ann_file = sorted(annotations_dir.glob("*.json"))[0]

with open(ann_file, 'r') as f:
    annotation = json.load(f)

print("Sample annotation structure:")
print(json.dumps(annotation, indent=2))

## 6. Compare Augmentations

In [ ]:
# Find images from the same original and compare augmentations
annotations_dir = OUTPUT_DIR / "annotations"
images_dir = OUTPUT_DIR / "images"

# Group by original image
originals = {}
for ann_file in annotations_dir.glob("*.json"):
    with open(ann_file, 'r') as f:
        ann = json.load(f)
    orig = ann["original_image"]
    if orig not in originals:
        originals[orig] = []
    originals[orig].append(ann)

# Pick first group with multiple augmentations
for orig, anns in originals.items():
    if len(anns) >= 3:
        anns.sort(key=lambda x: x["augmentation_index"])
        
        fig, axes = plt.subplots(1, len(anns), figsize=(6*len(anns), 6))
        
        for idx, ann in enumerate(anns):
            img = Image.open(images_dir / ann["image_name"])
            axes[idx].imshow(img)
            
            # Draw bboxes
            for space in ann["spaces"]:
                x1, y1, x2, y2 = space["bbox"]
                color = 'red' if space["occupied"] else 'lime'
                rect = Rectangle(
                    (x1, y1), x2-x1, y2-y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                axes[idx].add_patch(rect)
            
            title = "Original" if ann["augmentation_index"] == 0 else f"Aug {ann['augmentation_index']}"
            axes[idx].set_title(title)
            axes[idx].axis('off')
        
        plt.suptitle(f"{anns[0]['parking_lot']} - Augmentation Comparison")
        plt.tight_layout()
        plt.show()
        
        # Save the comparison plot
        if REPORT_UTILS_AVAILABLE:
            save_eda_plot(fig, "phase2_augmentation_comparison.png")
        
        break

## 7. Load Sample for Training

In [ ]:
# Example: Load a sample in a format ready for model training
def load_sample_for_training(output_dir, idx=0):
    """Load a sample ready for model training."""
    output_dir = Path(output_dir)
    images_dir = output_dir / "images"
    annotations_dir = output_dir / "annotations"
    
    ann_files = sorted(annotations_dir.glob("*.json"))
    
    with open(ann_files[idx], 'r') as f:
        ann = json.load(f)
    
    # Load image as numpy array
    img = Image.open(images_dir / ann["image_name"])
    img_array = np.array(img)
    
    # Extract bboxes and labels
    bboxes = np.array([s["bbox"] for s in ann["spaces"]])
    labels = np.array([1 if s["occupied"] else 0 for s in ann["spaces"]])
    
    return {
        "image": img_array,
        "bboxes": bboxes,
        "labels": labels
    }

# Test it
sample = load_sample_for_training(OUTPUT_DIR, idx=0)
print(f"Image shape: {sample['image'].shape}")
print(f"Bboxes shape: {sample['bboxes'].shape}")
print(f"Labels shape: {sample['labels'].shape}")
print(f"\nFirst 3 bboxes:\n{sample['bboxes'][:3]}")
print(f"\nFirst 3 labels:\n{sample['labels'][:3]}")

## Summary

This notebook demonstrated:
- Creating an augmented parking dataset
- Visualizing random samples
- Analyzing dataset statistics
- Comparing augmentation effects
- Loading data for model training

The dataset is now ready for training object detection or classification models!